# 🌳 Day 07: DP on Trees — Combining Recursion with DP

---

## What Is Tree DP?

DP where the subproblems are **subtrees**. Each node computes its answer from its children's answers (post-order DFS).

```
def dfs(node):
    left_result = dfs(node.left)
    right_result = dfs(node.right)
    return combine(node.val, left_result, right_result)
```

The recursion IS the DP. Each subtree is solved once (DFS visits each node once). The "table" is implicit in the call stack.

---

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    """Build tree from level-order list (None = no node)."""
    if not values:
        return None
    from collections import deque
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

## 🎯 Problem 1: House Robber III (LC #337)

Houses form a binary tree. Can't rob two directly-linked houses. Maximize money.

**At each node, two choices:**
- **Rob this node:** Take its value + skip children (use grandchildren's results)
- **Skip this node:** Take the best from both children (they might or might not be robbed)

**Return (rob_this, skip_this) pair from each subtree.**

In [ ]:
def rob_iii(root):
    """
    Tree DP: each node returns (rob_me, skip_me).
    Time: O(n), Space: O(h) call stack
    """
    def dfs(node):
        if not node:
            return (0, 0)  # (rob, skip)
        
        left = dfs(node.left)
        right = dfs(node.right)
        
        # Rob this node: can't rob children → use their skip values
        rob = node.val + left[1] + right[1]
        
        # Skip this node: children can be robbed or skipped (take best)
        skip = max(left) + max(right)
        
        return (rob, skip)
    
    return max(dfs(root))

root = build_tree([3, 2, 3, None, 3, None, 1])
print(rob_iii(root))  # 7 (3 + 3 + 1)

root2 = build_tree([3, 4, 5, 1, 3, None, 1])
print(rob_iii(root2))  # 9 (4 + 5)

**This is THE classic tree DP pattern:** return a tuple of choices from each subtree, let the parent combine them.

---

## 🎯 Problem 2: Binary Tree Maximum Path Sum (LC #124)

Find the maximum path sum (any path, not necessarily root-to-leaf).

**At each node:**
- **Gain from left/right:** `max(0, child_gain)` — negative gain = don't extend
- **Path through node:** `node.val + left_gain + right_gain` (potential answer)
- **Return to parent:** `node.val + max(left_gain, right_gain)` (can only extend one way up)

In [ ]:
def maxPathSum(root):
    """
    Tree DP with global max. Return max single-branch gain.
    Time: O(n), Space: O(h)
    """
    max_sum = float('-inf')
    
    def dfs(node):
        nonlocal max_sum
        if not node:
            return 0
        
        left_gain = max(0, dfs(node.left))   # Ignore negative branches
        right_gain = max(0, dfs(node.right))
        
        # Path through this node (could be the answer)
        max_sum = max(max_sum, node.val + left_gain + right_gain)
        
        # Return best single-branch gain to parent
        return node.val + max(left_gain, right_gain)
    
    dfs(root)
    return max_sum

root = build_tree([-10, 9, 20, None, None, 15, 7])
print(maxPathSum(root))  # 42 (15 + 20 + 7)

---

## 🎯 Problem 3: Longest ZigZag Path in Binary Tree (LC #1372)

A ZigZag path alternates left-right-left-right. Find the longest one.

**At each node, track:** how long the zigzag is if we came from left vs right.
- Going left next: extends a right-coming zigzag by 1
- Going right next: extends a left-coming zigzag by 1

In [ ]:
def longestZigZag(root):
    """
    Tree DP: return (left_zigzag_length, right_zigzag_length) from each node.
    Time: O(n), Space: O(h)
    """
    max_len = 0
    
    def dfs(node):
        """Returns (zigzag length going left, zigzag length going right)"""
        nonlocal max_len
        if not node:
            return (-1, -1)  # -1 so leaf returns (0, 0)
        
        left = dfs(node.left)    # (left_of_left, right_of_left)
        right = dfs(node.right)  # (left_of_right, right_of_right)
        
        # If I go left, I continue the right zigzag of my left child
        go_left = left[1] + 1
        # If I go right, I continue the left zigzag of my right child
        go_right = right[0] + 1
        
        max_len = max(max_len, go_left, go_right)
        return (go_left, go_right)
    
    dfs(root)
    return max_len

root = build_tree([1, None, 1, 1, 1, None, None, 1, 1, None, 1])
print(longestZigZag(root))  # 3

---

## 🗺️ Tree DP Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  TREE DP PATTERNS                                                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  RETURN TUPLE: each node returns multiple values                    ║
║    (rob, skip) — House Robber III                                   ║
║    (left_zz, right_zz) — ZigZag                                    ║
║                                                                      ║
║  GLOBAL MAX + RETURN: update global answer at each node,           ║
║    but return constrained value to parent                           ║
║    Max Path Sum: update with both branches, return one             ║
║                                                                      ║
║  GENERAL TEMPLATE:                                                  ║
║    def dfs(node):                                                   ║
║        left = dfs(node.left)                                        ║
║        right = dfs(node.right)                                      ║
║        update_global_answer(node, left, right)                      ║
║        return value_for_parent(node, left, right)                   ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 337 | House Robber III | Medium | Return (rob, skip) tuple |
| 124 | Binary Tree Max Path Sum | Hard | Global max + return single branch |
| 1372 | Longest ZigZag Path | Medium | Return (left_zz, right_zz) |
| 543 | Diameter of Binary Tree | Easy | Height DFS + global diameter |
| 968 | Binary Tree Cameras | Hard | Return state (covered, has_camera, needs) |
| 979 | Distribute Coins in Binary Tree | Medium | Excess/deficit per subtree |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Tree DP = post-order DFS where children inform parent          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Return tuples for multiple states per node                     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Global variable + constrained return is a common pattern       │
├──────────────────────────────────────────────────────────────────────┤
│  📌 max(0, child_gain) to ignore negative branches                 │
└──────────────────────────────────────────────────────────────────────┘
```

---

## 🎉 Chapter 06: Dynamic Programming — COMPLETE!

You've covered: fundamentals, 1D DP, 2D DP, string DP, knapsack, interval DP, and tree DP. That's every major DP pattern for interviews.

## ➡️ Next Chapter: Recursion & Backtracking